# MAJ-Debate: Stage 1 Side-Picking Agent Generation
**Authors:** Prajwal Bhandary, Saugat Shakya, Prabidhi Pyakurel, Rahul Shakya  
**Asian Institute of Technology (AIT), NLP Course (Master's Program)**

---
### Notebook Objective
This notebook implements Stage 1 of the MAJ-Debate pipeline: two-round argument generation using fixed persona agents under controlled prompt conditions.

### Scope of Work
1. **Persona Setup**: define fixed Pro and Con persona pools with explicit reasoning styles
2. **Sub-round 1 Generation**: produce independent stance-specific arguments without cross-side visibility
3. **Sub-round 2 Generation**: generate targeted counter-arguments conditioned on opposing Round 1 claims
4. **Artifact Export**: persist Stage 1 outputs for downstream evaluation and graph construction

### Reproducibility Notes
- Requires `GROQ_API_KEY` for live generation.
- Optionally logs experiments via `MLFLOW_TRACKING_URI`.
- Writes artifacts to project-level `outputs/stage1/` only (sibling of `notebooks/`).

## 0. Imports & Configuration
> **Set your keys here before running any other cell.**

In [1]:
import os, json, time, textwrap, logging
from pathlib import Path
from datetime import datetime
import urllib.request
import urllib.error
import requests
# ── Keys ──────────────────────────────────────────────────────────────────────
GROQ_API_KEY        = os.environ.get('GROQ_API_KEY', 'your-groq-api-key-here')
MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://YOUR_VM_IP:5000')

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL            = 'llama-3.3-70b-versatile'
TEMPERATURE      = 0.7
MAX_TOKENS       = 512
RATE_LIMIT_SLEEP = 1.2   # seconds between Groq calls

# ── Agent config ──────────────────────────────────────────────────────────────
N_PRO             = 3    # proposal default; set 2 for faster dev runs
N_CON             = 3
ARGS_PER_AGENT_R1 = 3    # Sub-round 1 args per agent
ARGS_PER_AGENT_R2 = 2    # Sub-round 2 counter-args per agent

# ── Resolve project root so outputs always go to <project>/outputs/stage1 ───
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists() and (p / 'outputs').exists()), cwd)

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR  = PROJECT_ROOT / 'outputs' / 'stage1'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / 'stage1_arguments.json'

USE_MOCK = (GROQ_API_KEY == 'your-groq-api-key-here')
MAX_RETRIES = 3
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('stage1')

print(f'Mode              : {"MOCK" if USE_MOCK else "LIVE (Groq)"}')
print(f'Model             : {MODEL}')
print(f'Agents            : {N_PRO} Pro + {N_CON} Con')
print(f'Args/agent R1/R2  : {ARGS_PER_AGENT_R1} / {ARGS_PER_AGENT_R2}')
print(f'MLflow URI        : {MLFLOW_TRACKING_URI}')
print(f'Project root      : {PROJECT_ROOT}')
print(f'Output            : {OUT_FILE.resolve()}')

Mode              : LIVE (Groq)
Model             : llama-3.3-70b-versatile
Agents            : 3 Pro + 3 Con
Args/agent R1/R2  : 3 / 2
MLflow URI        : http://85.211.242.44:5000
Project root      : F:\AIT\Jan 2026\NLU\Project
Output            : F:\AIT\Jan 2026\NLU\Project\outputs\stage1\stage1_arguments.json


## 1. Persona Definitions
Fixed across ALL experimental configurations — controlled independent variable for RQ2.  
Each persona specifies: `reasoning_style`, `rhetorical_mode`, `stance`.

In [2]:
PRO_PERSONAS = [
    {
        'id':              'pro_rationalist',
        'name':            'Rationalist Pro',
        'stance':          'PRO',
        'reasoning_style': 'logical-empirical',
        'rhetorical_mode': 'cite quantitative evidence and causal mechanisms',
        'description':     'Argues from data, statistics, and formal logic. Prioritises measurable outcomes.',
    },
    {
        'id':              'pro_ethicist',
        'name':            'Ethics Advocate Pro',
        'stance':          'PRO',
        'reasoning_style': 'ethical-normative',
        'rhetorical_mode': 'appeal to moral principles and rights-based frameworks',
        'description':     'Argues from fairness and justice. References established ethical frameworks.',
    },
    {
        'id':              'pro_futurist',
        'name':            'Futurist Pro',
        'stance':          'PRO',
        'reasoning_style': 'economic-consequentialist',
        'rhetorical_mode': 'project long-term societal and economic benefits',
        'description':     'Argues from systemic benefits and long-horizon impact. Accepts short-term trade-offs.',
    },
]

CON_PERSONAS = [
    {
        'id':              'con_skeptic',
        'name':            'Skeptic Con',
        'stance':          'CON',
        'reasoning_style': 'logical-empirical',
        'rhetorical_mode': 'challenge evidence quality and burden of proof',
        'description':     'Contests factual claims, demands rigorous evidence, identifies logical fallacies.',
    },
    {
        'id':              'con_rights',
        'name':            'Rights Defender Con',
        'stance':          'CON',
        'reasoning_style': 'ethical-normative',
        'rhetorical_mode': 'appeal to human rights, procedural justice, and democratic accountability',
        'description':     'Argues the proposal violates fundamental rights regardless of practical merits.',
    },
    {
        'id':              'con_pragmatist',
        'name':            'Pragmatist Con',
        'stance':          'CON',
        'reasoning_style': 'economic-consequentialist',
        'rhetorical_mode': 'highlight implementation barriers and unintended consequences',
        'description':     'Argues from practical constraints: cost, feasibility, second-order effects.',
    },
]

ACTIVE_PRO = PRO_PERSONAS[:N_PRO]
ACTIVE_CON = CON_PERSONAS[:N_CON]

print(f'Active Pro personas ({N_PRO}):')
for p in ACTIVE_PRO:
    print(f'  {p["name"]:30s} | {p["reasoning_style"]}')
print(f'Active Con personas ({N_CON}):')
for p in ACTIVE_CON:
    print(f'  {p["name"]:30s} | {p["reasoning_style"]}')

Active Pro personas (3):
  Rationalist Pro                | logical-empirical
  Ethics Advocate Pro            | ethical-normative
  Futurist Pro                   | economic-consequentialist
Active Con personas (3):
  Skeptic Con                    | logical-empirical
  Rights Defender Con            | ethical-normative
  Pragmatist Con                 | economic-consequentialist


## 2. Topics

In [3]:
TOPICS = [
    {'id':'T01','domain':'policy',  'text':'AI should replace human judges in courtrooms'},
    {'id':'T02','domain':'policy',  'text':'Universal basic income should be implemented globally'},
    {'id':'T03','domain':'policy',  'text':'Social media platforms should be regulated as public utilities'},
    {'id':'T04','domain':'ethics',  'text':'Gene editing in human embryos should be permitted for disease prevention'},
    {'id':'T05','domain':'ethics',  'text':'Autonomous weapons should be banned under international law'},
    {'id':'T06','domain':'ethics',  'text':'Companies have a moral obligation to prioritise employee wellbeing over profit'},
    {'id':'T07','domain':'science', 'text':'Nuclear energy is the best solution to climate change'},
    {'id':'T08','domain':'science', 'text':'Space colonisation should be a global priority over solving Earth problems'},
    {'id':'T09','domain':'society', 'text':'Social media does more harm than good to democratic discourse'},
    {'id':'T10','domain':'society', 'text':'Remote work should become the default for knowledge workers'},
]

print(f'Total topics: {len(TOPICS)}')
for t in TOPICS:
    print(f'  [{t["id"]}] ({t["domain"]:8s}) {t["text"]}')

Total topics: 10
  [T01] (policy  ) AI should replace human judges in courtrooms
  [T02] (policy  ) Universal basic income should be implemented globally
  [T03] (policy  ) Social media platforms should be regulated as public utilities
  [T04] (ethics  ) Gene editing in human embryos should be permitted for disease prevention
  [T05] (ethics  ) Autonomous weapons should be banned under international law
  [T06] (ethics  ) Companies have a moral obligation to prioritise employee wellbeing over profit
  [T07] (science ) Nuclear energy is the best solution to climate change
  [T08] (science ) Space colonisation should be a global priority over solving Earth problems
  [T09] (society ) Social media does more harm than good to democratic discourse
  [T10] (society ) Remote work should become the default for knowledge workers


## 3. Prompt Templates

In [4]:
def build_r1_prompt(topic, persona, n_args):
    """Sub-round 1: blind argument generation. Agent has NO knowledge of opposing arguments."""
    return (
        f'You are a debate agent with the following profile:\n'
        f'- Name            : {persona["name"]}\n'
        f'- Stance          : {persona["stance"]} (argue FOR this position only)\n'
        f'- Reasoning style : {persona["reasoning_style"]}\n'
        f'- Rhetorical mode : {persona["rhetorical_mode"]}\n'
        f'- Profile         : {persona["description"]}\n'
        f'\n'
        f'Debate topic: "{topic}"\n'
        f'\n'
        f'Generate exactly {n_args} distinct high-quality arguments for the {persona["stance"]} position.\n'
        f'Each argument must:\n'
        f'  1. Be a single clear sentence (max 40 words)\n'
        f'  2. Reflect your reasoning style and rhetorical mode\n'
        f'  3. Be substantively different from the others\n'
        f'\n'
        f'Output ONLY a valid JSON array of strings. No preamble, no markdown:\n'
        f'["argument 1", "argument 2", "argument 3"]'
    )


def build_r2_prompt(topic, persona, opposing_args, n_args):
    """Sub-round 2: counter-argument generation. Agent SEES opposing R1 arguments."""
    numbered = '\n'.join(f'  [{i+1}] {a}' for i, a in enumerate(opposing_args))
    return (
        f'You are a debate agent with the following profile:\n'
        f'- Name            : {persona["name"]}\n'
        f'- Stance          : {persona["stance"]}\n'
        f'- Reasoning style : {persona["reasoning_style"]}\n'
        f'- Rhetorical mode : {persona["rhetorical_mode"]}\n'
        f'\n'
        f'Debate topic: "{topic}"\n'
        f'\n'
        f'The opposing side has made these arguments (read-only context):\n'
        f'{numbered}\n'
        f'\n'
        f'Generate exactly {n_args} targeted counter-arguments from your {persona["stance"]} position.\n'
        f'Each counter-argument must:\n'
        f'  1. Directly attack one specific opposing argument above\n'
        f'  2. Be a single clear sentence (max 50 words)\n'
        f'  3. Reflect your reasoning style and rhetorical mode\n'
        f'\n'
        f'Output ONLY a valid JSON array of objects. No preamble, no markdown:\n'
        f'[{{"targets_arg": 1, "argument": "..."}}, {{"targets_arg": 3, "argument": "..."}}]'
    )


print('Prompts defined.')
print('\n--- R1 prompt preview ---')
print(build_r1_prompt(TOPICS[0]['text'], ACTIVE_PRO[0], ARGS_PER_AGENT_R1))

Prompts defined.

--- R1 prompt preview ---
You are a debate agent with the following profile:
- Name            : Rationalist Pro
- Stance          : PRO (argue FOR this position only)
- Reasoning style : logical-empirical
- Rhetorical mode : cite quantitative evidence and causal mechanisms
- Profile         : Argues from data, statistics, and formal logic. Prioritises measurable outcomes.

Debate topic: "AI should replace human judges in courtrooms"

Generate exactly 3 distinct high-quality arguments for the PRO position.
Each argument must:
  1. Be a single clear sentence (max 40 words)
  2. Reflect your reasoning style and rhetorical mode
  3. Be substantively different from the others

Output ONLY a valid JSON array of strings. No preamble, no markdown:
["argument 1", "argument 2", "argument 3"]


## 4. Groq API Client

In [5]:
GROQ_URL = 'https://api.groq.com/openai/v1/chat/completions'

def call_groq(prompt):
    """
    Single Groq API call using requests.
    Retries on 429 (rate limit) and 5xx errors.
    """
    headers = {
        'Authorization': f'Bearer {GROQ_API_KEY}',
        'Content-Type':  'application/json',
    }
    payload = {
        'model':       MODEL,
        'messages':    [{'role': 'user', 'content': prompt}],
        'temperature': TEMPERATURE,
        'max_tokens':  MAX_TOKENS,
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=45)
            if resp.status_code == 200:
                return resp.json()['choices'][0]['message']['content'].strip()
            elif resp.status_code == 429:
                wait = 5 * attempt
                log.warning('Rate limited — waiting %ds (attempt %d/%d)', wait, attempt, MAX_RETRIES)
                time.sleep(wait)
            elif resp.status_code >= 500:
                log.warning('Server error %d — retrying (attempt %d/%d)', resp.status_code, attempt, MAX_RETRIES)
                time.sleep(3)
            else:
                raise RuntimeError(f'Groq API error {resp.status_code}: {resp.text[:300]}')
        except requests.exceptions.Timeout:
            log.warning('Timeout (attempt %d/%d)', attempt, MAX_RETRIES)
            time.sleep(3)
        except requests.exceptions.ConnectionError as e:
            raise RuntimeError(f'Connection error: {e}') from e
    raise RuntimeError(f'Groq call failed after {MAX_RETRIES} attempts')


def clean_llm_text(text):
    """
    Normalise raw LLM output before JSON parsing.
    Handles the most common ways models break JSON:
      - markdown fences (```json ... ```)
      - smart/curly quotes instead of straight quotes
      - trailing commas before ] or }
      - leading/trailing prose outside the JSON array
    """
    import re
    text = text.strip()

    # Strip markdown fences
    if '```' in text:
        blocks = re.findall(r'```(?:json)?\s*([\s\S]*?)```', text)
        if blocks:
            text = blocks[0].strip()

    # Replace curly/smart quotes with straight quotes
    for bad, good in [(chr(0x201c), '"'), (chr(0x201d), '"'),
                      (chr(0x2018), "'"),  (chr(0x2019), "'")]:
        text = text.replace(bad, good)

    # Remove trailing commas before ] or } (invalid JSON)
    text = re.sub(r',\s*([\]}])', r'\1', text)

    # Isolate the first [ ... ] block if there is surrounding prose
    s = text.find('[')
    e = text.rfind(']') + 1
    if s != -1 and e > s:
        text = text[s:e]

    return text


def extract_strings_fallback(text):
    """
    Last-resort extractor when JSON is completely unparseable.
    Pulls out all double-quoted strings from the text.
    Works for R1 responses where we expect a flat list of strings.
    """
    import re
    # Match "..." allowing escaped quotes inside
    matches = re.findall(r'"((?:[^"\\]|\\.)*)"', text)
    # Filter out very short strings (likely JSON keys, not arguments)
    return [m for m in matches if len(m) > 20]


def parse_json_list(text, mode='r1'):
    """
    Robustly parse a JSON array from LLM output.

    mode='r1' expects list of strings
    mode='r2' expects list of {targets_arg, argument} dicts

    Falls back gracefully at each stage rather than raising.
    """
    import re

    cleaned = clean_llm_text(text)

    # Attempt 1: clean JSON parse
    try:
        result = json.loads(cleaned)
        if isinstance(result, list):
            log.info('JSON parsed cleanly (%d items)', len(result))
            return result
    except json.JSONDecodeError as err:
        log.warning('Clean JSON parse failed: %s — trying repairs', err)

    # Attempt 2: fix unescaped internal quotes by replacing " inside strings
    # Strategy: only keep the outermost quotes of each array element
    try:
        # For R1: model sometimes returns ["arg1", "arg2"] with bad inner quotes
        # Extract content between commas at the top level
        items = re.split(r'",\s*"', cleaned.strip('[]'))
        items = [item.strip().strip('"') for item in items]
        items = [i for i in items if len(i) > 15]
        if items:
            log.warning('Used comma-split fallback — got %d items', len(items))
            if mode == 'r1':
                return items
    except Exception:
        pass

    # Attempt 3: regex string extraction (R1 only)
    if mode == 'r1':
        items = extract_strings_fallback(text)
        if items:
            log.warning('Used regex string extraction — got %d items', len(items))
            return items

    # Attempt 4: for R2, try to find targets_arg + argument pairs via regex
    if mode == 'r2':
        pairs = re.findall(
            r'["\']?targets_arg["\']?\s*:\s*(\d+)[^}]*?["\']?argument["\']?\s*:\s*["\']([^"\']+)["\']',
            text, re.DOTALL
        )
        if pairs:
            result = [{'targets_arg': int(t), 'argument': a} for t, a in pairs]
            log.warning('Used R2 regex extraction — got %d pairs', len(result))
            return result

    log.error('All parse attempts failed. Raw text: %s', text[:200])
    return []


# ── Connectivity test ─────────────────────────────────────────────────────────
if not USE_MOCK:
    print('Testing Groq connectivity...')
    try:
        test_resp = requests.get(
            'https://api.groq.com/openai/v1/models',
            headers={'Authorization': f'Bearer {GROQ_API_KEY}'},
            timeout=10
        )
        if test_resp.status_code == 200:
            models = [m['id'] for m in test_resp.json().get('data', [])]
            if MODEL in models:
                print(f'Groq OK — model "{MODEL}" is available')
            else:
                print(f'Groq OK but "{MODEL}" not listed. Available: {models[:5]}')
        else:
            print(f'Groq returned {resp.status_code}: {resp.text[:200]}')
    except Exception as ex:
        print(f'Connectivity test failed: {ex}')
else:
    print('MOCK mode — skipping connectivity test')


Testing Groq connectivity...
Groq OK — model "llama-3.3-70b-versatile" is available


## 5. Mock Data
Used when `USE_MOCK=True` — realistic output for testing Stage 2 without API calls.

In [6]:
MOCK_R1 = {
    'T01': {
        'PRO': [
            'AI eliminates documented cognitive biases such as anchoring and confirmation bias that systematically distort human judicial reasoning.',
            'Algorithmic sentencing achieves consistent application of identical legal standards across demographically diverse defendants.',
            'Advances in explainable AI make algorithmic verdicts more interpretable and auditable than the opaque intuitions of human judges.',
        ],
        'CON': [
            'Justice requires contextual moral reasoning grounded in lived human experience that no current AI system can replicate.',
            'Defendants hold a fundamental right to be judged by their peers, and algorithmic sentencing removes democratic accountability.',
            'AI systems trained on historical case data encode and perpetuate historical racial and socioeconomic biases at scale.',
        ],
    }
}

MOCK_R2 = {
    'T01': {
        'PRO': [
            {'targets_arg': 3, 'argument': 'Counter to [3]: Training data bias is an engineering problem with known solutions such as fairness-aware learning, unlike structural human bias which is irreducible.'},
            {'targets_arg': 1, 'argument': 'Counter to [1]: Contextual reasoning can be approximated via structured persona prompts and case-law retrieval, making the experience argument empirically testable.'},
        ],
        'CON': [
            {'targets_arg': 1, 'argument': 'Counter to [1]: Eliminating bias assumes the bias taxonomy is complete and measurable, yet AI introduces novel failure modes such as distributional shift that are equally undetectable.'},
            {'targets_arg': 3, 'argument': 'Counter to [3]: Interpretability in AI remains a research problem without production-ready solutions, making auditability claims premature and potentially misleading to policymakers.'},
        ],
    }
}

print('Mock data ready for topics:', list(MOCK_R1.keys()))

Mock data ready for topics: ['T01']


## 6. Core Stage 1 Functions
### Sub-round 1 and Sub-round 2 runners

In [7]:
def run_subround1(topic, personas, n_args):
    """Sub-round 1: blind generation. No cross-side visibility."""
    results = {}
    for persona in personas:
        pid = persona['id']
        log.info('R1 | %s | %s', topic['id'], persona['name'])
        if USE_MOCK:
            pool = MOCK_R1.get(topic['id'], MOCK_R1.get('T01', {}))
            args = pool.get(persona['stance'], [])[:n_args]
        else:
            raw  = call_groq(build_r1_prompt(topic['text'], persona, n_args))
            log.debug('R1 raw response: %s', raw[:200])
            parsed = parse_json_list(raw, mode='r1')   # mode='r1' -> expect list of strings
            args = [str(a) for a in parsed if a][:n_args]
            time.sleep(RATE_LIMIT_SLEEP)
        results[pid] = {'persona': persona, 'arguments': args, 'round': 1}
        log.info('R1 done | %s | %d args', persona['name'], len(args))
    return results


def run_subround2(topic, personas, r1_opposing, n_args):
    """
    Sub-round 2: counter-argument generation.
    Personas receive r1_opposing (other side's R1 args) as read-only context.
    """
    opposing_args = []
    for v in r1_opposing.values():
        opposing_args.extend(v['arguments'])

    results = {}
    for persona in personas:
        pid = persona['id']
        log.info('R2 | %s | %s', topic['id'], persona['name'])
        if USE_MOCK:
            pool = MOCK_R2.get(topic['id'], MOCK_R2.get('T01', {}))
            counter_args = pool.get(persona['stance'], [])[:n_args]
        else:
            raw    = call_groq(build_r2_prompt(topic['text'], persona, opposing_args, n_args))
            log.debug('R2 raw response: %s', raw[:200])
            parsed = parse_json_list(raw, mode='r2')   # mode='r2' -> expect list of dicts
            counter_args = []
            for item in parsed[:n_args]:
                if isinstance(item, dict):
                    counter_args.append({
                        'targets_arg': item.get('targets_arg', -1),
                        'argument':    item.get('argument', str(item)),
                    })
                elif isinstance(item, str) and item:
                    counter_args.append({'targets_arg': -1, 'argument': item})
            time.sleep(RATE_LIMIT_SLEEP)
        results[pid] = {
            'persona':          persona,
            'counter_args':     counter_args,
            'opposing_context': opposing_args,
            'round':            2,
        }
        log.info('R2 done | %s | %d counter-args', persona['name'], len(counter_args))
    return results


print('Stage 1 runner functions ready.')


Stage 1 runner functions ready.


## 7. MLflow Setup

In [8]:
try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('MAJ-Debate')
    MLFLOW_OK = True
    print(f'MLflow connected: {MLFLOW_TRACKING_URI}')
except Exception as ex:
    MLFLOW_OK = False
    print(f'MLflow unavailable ({ex}) - results saved locally only')


def mlflow_log(run_name, params, metrics, artifact_paths):
    if not MLFLOW_OK:
        return
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for path in artifact_paths:
            mlflow.log_artifact(str(path), artifact_path='stage1')
    print(f'MLflow run logged: {run_name}')

MLflow unavailable (No module named 'pkg_resources') - results saved locally only


## 8. Main Run - Stage 1 on All Topics

In [9]:
RUN_TS   = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'stage1-{RUN_TS}'
all_results   = []
total_r1_args = 0
total_r2_args = 0

print(f'Run: {RUN_NAME}')
print(f'Topics: {len(TOPICS)}  |  Mode: {"MOCK" if USE_MOCK else "LIVE"}')
print('=' * 60)

for i, topic in enumerate(TOPICS):
    tid = topic['id']
    print(f'\n[{i+1}/{len(TOPICS)}] {tid}: {topic["text"]}')

    # Sub-round 1 ──────────────────────────────────────────────────────────────
    print('  Sub-round 1: independent generation...')
    pro_r1 = run_subround1(topic, ACTIVE_PRO, ARGS_PER_AGENT_R1)
    con_r1 = run_subround1(topic, ACTIVE_CON, ARGS_PER_AGENT_R1)
    r1_pro = sum(len(v['arguments']) for v in pro_r1.values())
    r1_con = sum(len(v['arguments']) for v in con_r1.values())
    print(f'  R1: Pro={r1_pro} args, Con={r1_con} args')

    # Sub-round 2 ──────────────────────────────────────────────────────────────
    print('  Sub-round 2: counter-argument generation...')
    pro_r2 = run_subround2(topic, ACTIVE_PRO, r1_opposing=con_r1, n_args=ARGS_PER_AGENT_R2)
    con_r2 = run_subround2(topic, ACTIVE_CON, r1_opposing=pro_r1, n_args=ARGS_PER_AGENT_R2)
    r2_pro = sum(len(v['counter_args']) for v in pro_r2.values())
    r2_con = sum(len(v['counter_args']) for v in con_r2.values())
    print(f'  R2: Pro={r2_pro} counter-args, Con={r2_con} counter-args')

    # Flatten to a list Stage 2 can consume ────────────────────────────────────
    flat_args = []
    idx = 0

    for pid, d in pro_r1.items():
        for arg in d['arguments']:
            flat_args.append({'arg_id': f'{tid}_A{idx:03d}', 'persona_id': pid,
                              'persona': d['persona']['name'], 'stance': 'PRO',
                              'round': 1, 'targets_arg': None, 'text': arg})
            idx += 1

    for pid, d in con_r1.items():
        for arg in d['arguments']:
            flat_args.append({'arg_id': f'{tid}_A{idx:03d}', 'persona_id': pid,
                              'persona': d['persona']['name'], 'stance': 'CON',
                              'round': 1, 'targets_arg': None, 'text': arg})
            idx += 1

    for pid, d in pro_r2.items():
        for ca in d['counter_args']:
            text = ca['argument'] if isinstance(ca, dict) else ca
            tgt  = ca.get('targets_arg') if isinstance(ca, dict) else None
            flat_args.append({'arg_id': f'{tid}_A{idx:03d}', 'persona_id': pid,
                              'persona': d['persona']['name'], 'stance': 'PRO',
                              'round': 2, 'targets_arg': tgt, 'text': text})
            idx += 1

    for pid, d in con_r2.items():
        for ca in d['counter_args']:
            text = ca['argument'] if isinstance(ca, dict) else ca
            tgt  = ca.get('targets_arg') if isinstance(ca, dict) else None
            flat_args.append({'arg_id': f'{tid}_A{idx:03d}', 'persona_id': pid,
                              'persona': d['persona']['name'], 'stance': 'CON',
                              'round': 2, 'targets_arg': tgt, 'text': text})
            idx += 1

    all_results.append({
        'topic_id':   tid,
        'topic_text': topic['text'],
        'domain':     topic['domain'],
        'run_name':   RUN_NAME,
        'arguments':  flat_args,
        'meta': {
            'n_pro': N_PRO, 'n_con': N_CON,
            'r1_per_agent': ARGS_PER_AGENT_R1,
            'r2_per_agent': ARGS_PER_AGENT_R2,
            'total_args': len(flat_args),
            'r1_args': r1_pro + r1_con,
            'r2_args': r2_pro + r2_con,
            'model': MODEL, 'mode': 'mock' if USE_MOCK else 'live',
        }
    })
    total_r1_args += r1_pro + r1_con
    total_r2_args += r2_pro + r2_con
    print(f'  Total args: {len(flat_args)} (R1: {r1_pro+r1_con}, R2: {r2_pro+r2_con})')

print(f'\n{"="*60}')
print(f'Stage 1 complete | Total R1: {total_r1_args} | Total R2: {total_r2_args} | Grand total: {total_r1_args+total_r2_args}')

2026-03-21 12:51:31,168 INFO R1 | T01 | Rationalist Pro


Run: stage1-20260321_125131
Topics: 10  |  Mode: LIVE

[1/10] T01: AI should replace human judges in courtrooms
  Sub-round 1: independent generation...


2026-03-21 12:51:32,045 INFO JSON parsed cleanly (3 items)
2026-03-21 12:51:33,251 INFO R1 done | Rationalist Pro | 3 args
2026-03-21 12:51:33,251 INFO R1 | T01 | Ethics Advocate Pro
2026-03-21 12:51:34,107 INFO JSON parsed cleanly (3 items)
2026-03-21 12:51:35,311 INFO R1 done | Ethics Advocate Pro | 3 args
2026-03-21 12:51:35,311 INFO R1 | T01 | Futurist Pro
2026-03-21 12:51:36,315 INFO JSON parsed cleanly (3 items)
2026-03-21 12:51:37,517 INFO R1 done | Futurist Pro | 3 args
2026-03-21 12:51:37,517 INFO R1 | T01 | Skeptic Con
2026-03-21 12:51:38,378 INFO JSON parsed cleanly (3 items)
2026-03-21 12:51:39,581 INFO R1 done | Skeptic Con | 3 args
2026-03-21 12:51:39,581 INFO R1 | T01 | Rights Defender Con
2026-03-21 12:51:40,495 INFO JSON parsed cleanly (3 items)
2026-03-21 12:51:41,697 INFO R1 done | Rights Defender Con | 3 args
2026-03-21 12:51:41,698 INFO R1 | T01 | Pragmatist Con
2026-03-21 12:51:42,528 INFO JSON parsed cleanly (3 items)
2026-03-21 12:51:43,729 INFO R1 done | Pragma

  R1: Pro=9 args, Con=9 args
  Sub-round 2: counter-argument generation...


2026-03-21 12:51:44,681 INFO JSON parsed cleanly (2 items)
2026-03-21 12:51:45,882 INFO R2 done | Rationalist Pro | 2 counter-args
2026-03-21 12:51:45,884 INFO R2 | T01 | Ethics Advocate Pro
2026-03-21 12:51:46,716 INFO JSON parsed cleanly (2 items)
2026-03-21 12:51:47,919 INFO R2 done | Ethics Advocate Pro | 2 counter-args
2026-03-21 12:51:47,919 INFO R2 | T01 | Futurist Pro
2026-03-21 12:51:48,722 INFO JSON parsed cleanly (2 items)
2026-03-21 12:51:49,928 INFO R2 done | Futurist Pro | 2 counter-args
2026-03-21 12:51:49,928 INFO R2 | T01 | Skeptic Con
2026-03-21 12:51:51,058 INFO JSON parsed cleanly (2 items)
2026-03-21 12:51:52,261 INFO R2 done | Skeptic Con | 2 counter-args
2026-03-21 12:51:52,263 INFO R2 | T01 | Rights Defender Con
2026-03-21 12:51:53,171 INFO JSON parsed cleanly (2 items)
2026-03-21 12:51:54,373 INFO R2 done | Rights Defender Con | 2 counter-args
2026-03-21 12:51:54,374 INFO R2 | T01 | Pragmatist Con
2026-03-21 12:51:55,355 INFO JSON parsed cleanly (2 items)
2026-

...

  R2: Pro=6 counter-args, Con=6 counter-args
  Total args: 30 (R1: 18, R2: 12)

[10/10] T10: Remote work should become the default for knowledge workers
  Sub-round 1: independent generation...


2026-03-21 12:55:20,861 INFO JSON parsed cleanly (3 items)
2026-03-21 12:55:22,063 INFO R1 done | Rationalist Pro | 3 args
2026-03-21 12:55:22,064 INFO R1 | T10 | Ethics Advocate Pro
2026-03-21 12:55:22,857 INFO JSON parsed cleanly (3 items)
2026-03-21 12:55:24,060 INFO R1 done | Ethics Advocate Pro | 3 args
2026-03-21 12:55:24,061 INFO R1 | T10 | Futurist Pro
2026-03-21 12:55:24,980 INFO JSON parsed cleanly (3 items)
2026-03-21 12:55:26,182 INFO R1 done | Futurist Pro | 3 args
2026-03-21 12:55:26,183 INFO R1 | T10 | Skeptic Con
2026-03-21 12:55:27,159 INFO JSON parsed cleanly (3 items)
2026-03-21 12:55:28,361 INFO R1 done | Skeptic Con | 3 args
2026-03-21 12:55:28,361 INFO R1 | T10 | Rights Defender Con
2026-03-21 12:55:29,306 INFO JSON parsed cleanly (3 items)
2026-03-21 12:55:30,507 INFO R1 done | Rights Defender Con | 3 args
2026-03-21 12:55:30,507 INFO R1 | T10 | Pragmatist Con
2026-03-21 12:55:31,341 INFO JSON parsed cleanly (3 items)
2026-03-21 12:55:32,543 INFO R1 done | Pragma

  R1: Pro=9 args, Con=9 args
  Sub-round 2: counter-argument generation...


2026-03-21 12:55:33,642 INFO JSON parsed cleanly (2 items)
2026-03-21 12:55:34,843 INFO R2 done | Rationalist Pro | 2 counter-args
2026-03-21 12:55:34,844 INFO R2 | T10 | Ethics Advocate Pro
2026-03-21 12:55:35,697 INFO JSON parsed cleanly (2 items)
2026-03-21 12:55:36,899 INFO R2 done | Ethics Advocate Pro | 2 counter-args
2026-03-21 12:55:36,900 INFO R2 | T10 | Futurist Pro
2026-03-21 12:55:37,790 INFO JSON parsed cleanly (2 items)
2026-03-21 12:55:38,992 INFO R2 done | Futurist Pro | 2 counter-args
2026-03-21 12:55:38,992 INFO R2 | T10 | Skeptic Con
2026-03-21 12:55:40,120 INFO JSON parsed cleanly (2 items)
2026-03-21 12:55:41,321 INFO R2 done | Skeptic Con | 2 counter-args
2026-03-21 12:55:41,321 INFO R2 | T10 | Rights Defender Con
2026-03-21 12:55:42,353 INFO JSON parsed cleanly (2 items)
2026-03-21 12:55:43,553 INFO R2 done | Rights Defender Con | 2 counter-args
2026-03-21 12:55:43,554 INFO R2 | T10 | Pragmatist Con
2026-03-21 12:55:44,381 INFO JSON parsed cleanly (2 items)
2026-

  R2: Pro=6 counter-args, Con=6 counter-args
  Total args: 30 (R1: 18, R2: 12)

Stage 1 complete | Total R1: 178 | Total R2: 120 | Grand total: 298


## 9. Save Output + Log to MLflow

In [10]:
output_doc = {
    'stage':     1,
    'run_name':  RUN_NAME,
    'timestamp': RUN_TS,
    'config': {
        'model': MODEL, 'n_pro': N_PRO, 'n_con': N_CON,
        'r1_per_agent': ARGS_PER_AGENT_R1, 'r2_per_agent': ARGS_PER_AGENT_R2,
        'mode': 'mock' if USE_MOCK else 'live',
    },
    'personas':  {'pro': ACTIVE_PRO, 'con': ACTIVE_CON},
    'topics':    all_results,
    'summary': {
        'total_topics':       len(TOPICS),
        'total_r1_args':      total_r1_args,
        'total_r2_args':      total_r2_args,
        'total_args':         total_r1_args + total_r2_args,
        'avg_args_per_topic': round((total_r1_args + total_r2_args) / len(TOPICS), 1),
        'r2_coverage_pct':    round(total_r2_args / (total_r1_args + total_r2_args) * 100, 1),
    }
}

with open(OUT_FILE, 'w') as f:
    json.dump(output_doc, f, indent=2)
print(f'Saved: {OUT_FILE.resolve()} ({OUT_FILE.stat().st_size/1024:.1f} KB)')

persona_file = OUT_DIR / 'personas.json'
with open(persona_file, 'w') as f:
    json.dump({'pro': ACTIVE_PRO, 'con': ACTIVE_CON}, f, indent=2)

mlflow_log(
    run_name=RUN_NAME,
    params={
        'model': MODEL, 'n_pro': N_PRO, 'n_con': N_CON,
        'r1_per_agent': ARGS_PER_AGENT_R1, 'r2_per_agent': ARGS_PER_AGENT_R2,
        'n_topics': len(TOPICS), 'temperature': TEMPERATURE,
        'mode': 'mock' if USE_MOCK else 'live',
    },
    metrics={
        'total_r1_args':      float(total_r1_args),
        'total_r2_args':      float(total_r2_args),
        'total_args':         float(total_r1_args + total_r2_args),
        'avg_args_per_topic': (total_r1_args + total_r2_args) / len(TOPICS),
        'r2_coverage_pct':    total_r2_args / (total_r1_args + total_r2_args) * 100,
    },
    artifact_paths=[OUT_FILE, persona_file],
)

print('\nSummary:')
for k, v in output_doc['summary'].items():
    print(f'  {k:<25} {v}')

Saved: F:\AIT\Jan 2026\NLU\Project\outputs\stage1\stage1_arguments.json (103.8 KB)

Summary:
  total_topics              10
  total_r1_args             178
  total_r2_args             120
  total_args                298
  avg_args_per_topic        29.8
  r2_coverage_pct           40.3


## 10. Inspect Output

In [11]:
# Print one topic in full to verify structure before passing to Stage 2
sample = all_results[0]
print(f'Topic : {sample["topic_text"]}')
print(f'Domain: {sample["domain"]}  |  Total args: {sample["meta"]["total_args"]}')
print()

for rnd in [1, 2]:
    rnd_args = [a for a in sample['arguments'] if a['round'] == rnd]
    print(f'--- Sub-round {rnd} ({len(rnd_args)} arguments) ---')
    for a in rnd_args:
        tgt = f" -> targets opp. arg #{a['targets_arg']}" if a['targets_arg'] else ''
        print(f"  [{a['stance']}] {a['persona']:25s}{tgt}")
        print(f"        {textwrap.fill(a['text'], 80, subsequent_indent='        ')}")
    print()

print('Stage 2 input schema (first argument):')
print(json.dumps(sample['arguments'][0], indent=2))

Topic : AI should replace human judges in courtrooms
Domain: policy  |  Total args: 30

--- Sub-round 1 (18 arguments) ---
  [PRO] Rationalist Pro          
        AI judges reduce sentencing disparities by 25% via algorithmic consistency.
  [PRO] Rationalist Pro          
        Machine learning predicts case outcomes with 87% accuracy, outperforming humans.
  [PRO] Rationalist Pro          
        Automated judges process 300% more cases per hour than human counterparts.
  [PRO] Ethics Advocate Pro      
        AI judges uphold fairness by eliminating biases
  [PRO] Ethics Advocate Pro      
        Machines ensure equal justice under the law
  [PRO] Ethics Advocate Pro      
        Replacing humans with AI judges respects the right to impartial trials
  [PRO] Futurist Pro             
        AI judges reduce long-term litigation costs through increased efficiency
  [PRO] Futurist Pro             
        Replacing human judges with AI enhances societal fairness by minimizing b

---
## Stage 1 Completion Summary

| Artifact | Path | Purpose |
|---|---|---|
| Stage 1 arguments | `outputs/stage1/stage1_arguments.json` | Primary Stage 1 output for downstream processing |
| Persona definitions | `outputs/stage1/personas.json` | Reproducible persona registry for experiment traceability |
| MLflow run | `MAJ-Debate` experiment (`stage1-<timestamp>`) | Parameter and metric tracking |

### Interface Contract for Downstream Stages
- `output_doc['topics']`: list of topic-level records
- Each topic contains `arguments`: flat list with `arg_id`, `stance`, `round`, `text`, and `targets_arg`

Re-run Stage 1 only when changing agent counts, generation parameters, topic set, or model configuration.